In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

In [4]:
### Load the all model - trained model, scaler pickle, onehot using Tesnorboard extension
model= load_model('churn_model.h5')

# Load the pickled encoders and scalers
with open('onehot_encoder_geo.pkl', 'rb') as f:
    onehot_encoder_geo = pickle.load(f)
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [21]:
#Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 25,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

# Process to train model to predict
1. convert Categorical data to Numerical
2. Apply std scaling
3. Apply prediction with model

In [22]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\HD\AI_Projects\MachineLearningNLP\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [23]:
# Convert the input data into a DataFrame
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,25,3,60000,2,1,1,50000


In [24]:
#convert the gender categorical data into numerical data using label encoder
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,25,3,60000,2,1,1,50000


In [25]:
# concatenate the original input data with the encoded geographical features
input_data_df = pd.concat([input_data_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,25,3,60000,2,1,1,50000,1.0,0.0,0.0


In [26]:
# scale the input data using the loaded scaler
input_data_scaled = scaler.transform(input_data_df)
input_data_scaled

array([[-0.53598516,  0.91324755, -1.32129295, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [27]:
#Predict the probability of churn using the loaded model
prediction = model.predict(input_data_scaled)
prediction
print(f"Predicted probability of churn: {prediction[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Predicted probability of churn: 0.0871


In [28]:
if prediction[0][0] > 0.5:
    print("The customer is likely to churn.")
else:    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
